In [ ]:
import json
import dataloader
from models.DeitModel import DeiTModel
from models.InceptionModel import InceptionModel
from models.MobileNetModel import MobileNetModel
from models.VGG16 import VGG16
import trainer
from models.MobileVIT import MobileVIT
from results import Results as results
from utils import Utilities as utils
import torch
import torchvision
from torch.utils.data import random_split
from torchsummary import summary
from transformers import ViTImageProcessor, ViTForImageClassification
from tqdm.auto import tqdm
from sklearn.decomposition import PCA
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.svm import SVC

In [ ]:
config = utils.get_config()
dataset_path = config['dataset_path']
dataset_name = config['dataset_name']
epochs = config['epochs']
train_size = config['train_size']
val_size = config['val_size']
learning_rate = config['learning_rate']
batch_size = config['batch_size']
results_file = config['results_file']
results_image = config['results_image']
save_model_file = config['save_model_file']
saved_weights = config['saved_weights']
model_input_shape = tuple(config['model_input_shape'])
model_name = config['model_name']
device = config['device']
current_run_dir = results.create_new_result_dir()

In [ ]:
processor = ViTImageProcessor.from_pretrained('google/vit-base-patch16-224') # Define a custom transform function using the processor
def transform(image): # Convert PIL image to format compatible with processor 
    inputs = processor(images=image, return_tensors="pt") 
    return inputs['pixel_values'].squeeze(0)

In [ ]:
if dataset_name.lower() == "FabricsDataset".lower():
    dataset = dataloader.FabricDataset(dataset_path, transform)
elif dataset_name.lower() == "FabricsOCTDataset".lower():
    dataset = dataloader.FabricOCTDataset(dataset_path, transform)
else:
    assert False, "Dataset name should be either FabricsDataset or FabricsOCTDataset"

In [ ]:
train_dataset, val_dataset = random_split(dataset, [train_size, val_size])
train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = torch.utils.data.DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

In [ ]:
train_model = ViTForImageClassification.from_pretrained('google/vit-base-patch16-224')
train_model.classifier = torch.nn.Sequential(
            torch.nn.Linear(768, 3),
            torch.nn.Softmax()
        )

In [ ]:
loss_fn = torch.nn.BCELoss()
optimizer = torch.optim.SGD(train_model.parameters(), lr=learning_rate)

In [ ]:
train_model

In [ ]:
train_runner = trainer.Trainer(train_model,
                                    loss_fn, 
                                    optimizer, 
                                    epochs, 
                                    train_loader, 
                                    val_loader, 
                                    log_results_file=f"{current_run_dir}/{results_file}",
                                    save_model_file=f"{current_run_dir}/{save_model_file}"
                                    )
result = train_runner.train()
results.save_acc_loss_graph(f"{currengt_run_dir}/{results_image}", *result)

In [ ]:
# model = torch.load("D:\Mayank\Thesis\Fabric\Results\Fabrics_Dataset\Split80+20\\12-11-2024\model-epoch-300", map_location=torch.device('cpu'))
# model.classifier = torch.nn.Identity()

In [ ]:
model = train_model
model.classifier = torch.nn.Identity()

In [ ]:
model

In [ ]:
# Function to preprocess and extract features from an image
def extract_features(image):
    image = image - torch.min(image)
    image = image / torch.max(image)
    inputs = processor(images=image, return_tensors="pt")
    with torch.no_grad():
        outputs = model(**inputs)
    outputs = outputs.logits.squeeze().mean(dim=0).numpy()
    # outputs = outputs.logits.squeeze().mean(dim=0).numpy()
    return outputs
    # hidden_states = outputs.hidden_states[-1] # Use the last hidden state
    # return hidden_states.squeeze().mean(dim=0).numpy()


In [ ]:
tsne_data, _ = random_split(dataset, [0.01, 0.99])

In [ ]:
len(tsne_data)

In [ ]:
count = [0, 0, 0]
features_list = []
labels_list = []
for i, data in tqdm(enumerate(tsne_data), total=len(tsne_data)):
    images, labels = data
    # print(images.shape, labels)
    features_list.append(extract_features(images))
    labels_list.append(torch.argmax(labels))
    count[torch.argmax(labels)] += 1
features_list = np.array(features_list)
labels_list = np.array(labels_list)
count
    

In [ ]:
# Ensure features are a 2D array
features_2d = features_list.reshape(len(features_list), -1)
tsne = TSNE(n_components=3, random_state=42)
features_2d_tsne = tsne.fit_transform(features_2d)

In [ ]:
label_names = ['Cotton', 'Polyester', 'Wool']
unique_labels = np.unique(labels_list)

plt.figure(figsize=(12, 8))
scatter = plt.scatter(features_2d_tsne[:, 0], features_2d_tsne[:, 1], c=labels_list, cmap='viridis', s=50)
plt.colorbar(scatter)
plt.title('t-SNE Visualization of ViT Features')
plt.xlabel('t-SNE Component 1')
plt.ylabel('t-SNE Component 2')

handles = [plt.Line2D([0], [0], marker='o', color='w', label=label_names[label], markersize=10, markerfacecolor=scatter.cmap(scatter.norm(label))) for label in unique_labels]
plt.legend(handles=handles, title="Labels", bbox_to_anchor=(0.01, 0.8), loc='lower left')

plt.show()

In [ ]:
# Apply PCA
pca = PCA(n_components=0.99)  # Adjust the number of components
pca_features = pca.fit_transform(features_2d)

In [ ]:
pca_features.shape

In [ ]:
# Apply LDA
lda = LinearDiscriminantAnalysis(n_components=3)  # Adjust the number of components
lda_features = lda.fit_transform(pca_features, labels_list)

In [ ]:
lda_features.shape

In [ ]:
# Train SVM with RBF kernel and C=1
svm = SVC(kernel='rbf', C=1, probability=True)
svm.fit(lda_features, labels)

# Make predictions
predictions = svm.predict(lda_features)
accuracy = accuracy_score(labels, predictions)
print(f'Accuracy: {accuracy}')